# Valora AI - Production Fine-Tuning (DeepSeek R1 7B)

**Enterprise Real Estate Intelligence for Bangalore**

## 📊 Dataset v3.3 (8,061 examples)
| Metric | Value |
|--------|-------|
| **Total Examples** | 8,061 |
| **Train** | 7,254 (90%) |
| **Eval** | 807 (10%) |
| **Categories** | 37 intent types |

### Business-Aligned Distribution
| Segment | % | Key Categories |
|---------|---|----------------|
| **Brokers** | 25% | PROPERTY_SEARCH, RECOMMENDATION, COMPARISON, REPORT_GEN |
| **Developers** | 25% | SIMULATE, SITE_SELECTION, CORRIDOR_ANALYSIS, ROI_ANALYSIS |
| **Banks/Lenders** | 20% | RISK_ASSESSMENT, CAP_RATE, RENTAL_YIELD, PORTFOLIO |
| **NRI/Remote** | 15% | NRI_QUERIES, ANALYZE_AREA, INVESTMENT, MULTILINGUAL |
| **Platform** | 15% | SPATIAL_3D, MULTI_TURN, REFUSAL, TROUBLESHOOT |

### 🆕 City Intelligence Sandbox (NEW)
| Category | Count | Description |
|----------|-------|-------------|
| **SIMULATE_SCENARIO** | 362 | What-if simulations with scenario deltas |
| **STORYBOARD** | 301 | 3D cinematic camera sequences & narration |
| **UI_CONTROL** | 301 | Panel/tab/mode control with ui_actions |
| **NARRATIVE** | 241 | Storytelling sequences for 3D visualization |

##  Quality Features
- **Truth Firewall**: Grounded facts only, no hallucination
- **` ` Tags**: Step-by-step reasoning (DeepSeek R1 native)
- **ui_actions JSON**: Frontend control commands
- **Storyboard JSON**: Camera positions, narration scripts
- **Scenario Deltas**: What-if impact projections

## ⚙️ Training Config
- **Model**: DeepSeek R1 7B (Qwen-based)
- **LoRA Rank**: 128 | **Alpha**: 256
- **Context**: 3,072 tokens | **QLoRA 4-bit**
- **Epochs**: 3 | **Batch**: 8 (effective)
- **LR**: 1e-5 with cosine schedule

## 🔄 Checkpoint Persistence
- Saves to `/kaggle/working/checkpoints/` every 100 steps
- Auto-resumes from last checkpoint on kernel restart
- Compatible with Kaggle persistent storage

In [ ]:
# Cell 1: Configuration with Persistent Storage
import os
from pathlib import Path
import shutil

print("🚀 KAGGLE FINE-TUNING CONFIG - DEEPSEEK R1 7B")
print("=" * 60)

# ===== MODEL CONFIG =====
MODEL_NAME = "unsloth/DeepSeek-R1-Distill-Qwen-7B"
MODEL_SIZE = "7B"
MAX_SEQ_LENGTH = 3072
LORA_RANK = 128
LORA_ALPHA = 256  # 2x rank for better learning

# ===== TRAINING CONFIG (RECOMMENDED SETTINGS) =====
NUM_EPOCHS = 3
LEARNING_RATE = 1e-5  # Conservative for stable training
BATCH_SIZE = 1
GRAD_ACCUM = 8  # Effective batch = 8
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 0.3

# ===== CHECKPOINT CONFIG (KAGGLE PERSISTENT) =====
# Use /kaggle/working for persistence across sessions
CHECKPOINT_DIR = Path("/kaggle/working/checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Final output directory
OUTPUT_DIR = Path("/kaggle/working/final_model")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Training data location
PROJECT_ROOT = Path("/kaggle/input/valora-training-data")
TRAIN_FILE = PROJECT_ROOT / "train.json"
EVAL_FILE = PROJECT_ROOT / "eval.json"

# ===== CHECKPOINT DETECTION =====
def find_latest_checkpoint():
    """Find the latest checkpoint for auto-resume"""
    checkpoints = sorted(CHECKPOINT_DIR.glob("checkpoint-*"), 
                        key=lambda x: int(x.name.split("-")[1]))
    if checkpoints:
        return str(checkpoints[-1])
    return None

latest_ckpt = find_latest_checkpoint()

print(f"Model: {MODEL_NAME}")
print(f"Max Sequence: {MAX_SEQ_LENGTH}")
print(f"LoRA: rank={LORA_RANK}, alpha={LORA_ALPHA}")
print(f"Training: {NUM_EPOCHS} epochs, lr={LEARNING_RATE}, batch={GRAD_ACCUM}")
print(f"Checkpoints: {CHECKPOINT_DIR}")
if latest_ckpt:
    print(f"🔄 RESUME FROM: {Path(latest_ckpt).name}")
else:
    print("🆕 Starting fresh training")
print("=" * 60)

In [ ]:
# Cell 2: Progress Tracker (Replaces Keep-Alive)
import threading
import time
import json
from datetime import datetime

class TrainingProgressTracker:
    """Track training progress and save state for resume"""
    
    def __init__(self, checkpoint_dir):
        self.checkpoint_dir = Path(checkpoint_dir)
        self.progress_file = self.checkpoint_dir / "training_progress.json"
        self.start_time = time.time()
        self.running = False
        self.thread = None
        self.current_step = 0
        self.total_steps = 0
        self.current_epoch = 0
        self.last_loss = None
        
    def _background_logger(self):
        """Background thread for progress logging"""
        while self.running:
            elapsed = time.time() - self.start_time
            hours = int(elapsed // 3600)
            minutes = int((elapsed % 3600) // 60)
            
            status = f"⏱️ {hours:02d}:{minutes:02d}"
            if self.total_steps > 0:
                pct = (self.current_step / self.total_steps) * 100
                status += f" | Step {self.current_step}/{self.total_steps} ({pct:.1f}%)"
            if self.last_loss:
                status += f" | Loss: {self.last_loss:.4f}"
            
            print(f"\r{status}", end='', flush=True)
            time.sleep(60)  # Log every minute
    
    def start(self, total_steps=0):
        """Start progress tracking"""
        self.total_steps = total_steps
        self.running = True
        self.thread = threading.Thread(target=self._background_logger, daemon=True)
        self.thread.start()
        print("✅ Progress tracker started")
    
    def update(self, step, loss=None, epoch=None):
        """Update current progress"""
        self.current_step = step
        if loss: self.last_loss = loss
        if epoch: self.current_epoch = epoch
        
        # Save progress to file for persistence
        self._save_progress()
    
    def _save_progress(self):
        """Save progress to JSON for resume capability"""
        progress = {
            "current_step": self.current_step,
            "total_steps": self.total_steps,
            "current_epoch": self.current_epoch,
            "last_loss": self.last_loss,
            "elapsed_seconds": time.time() - self.start_time,
            "timestamp": datetime.now().isoformat()
        }
        with open(self.progress_file, 'w') as f:
            json.dump(progress, f, indent=2)
    
    def load_progress(self):
        """Load previous progress if exists"""
        if self.progress_file.exists():
            with open(self.progress_file, 'r') as f:
                return json.load(f)
        return None
    
    def stop(self):
        """Stop tracking"""
        self.running = False
        if self.thread:
            self.thread.join(timeout=2)
        print("\n✅ Progress tracker stopped")

# Initialize tracker
progress_tracker = TrainingProgressTracker(CHECKPOINT_DIR)

# Check for previous progress
prev_progress = progress_tracker.load_progress()
if prev_progress:
    print(f"📊 Previous session: Step {prev_progress['current_step']}, Loss {prev_progress.get('last_loss', 'N/A')}")
else:
    print("📊 No previous progress found")

In [ ]:
# Cell 3: Install Dependencies (Optimized)
print("📦 Installing dependencies...")

# Core ML libraries
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q
!pip install transformers accelerate -q

# Training libraries
!pip install peft bitsandbytes trl -q
!pip install datasets pandas tqdm -q

# Unsloth for fast training
import gc; gc.collect()
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q

# Set HuggingFace timeout environment variables
import os
os.environ['HF_HUB_ETAG_TIMEOUT'] = '60'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '60'
os.environ['REQUESTS_TIMEOUT'] = '60'

# Verify GPU
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"✅ GPU: {gpu_name} ({gpu_mem:.1f} GB VRAM)")
else:
    print("❌ No GPU detected!")
    raise RuntimeError("GPU required for training")

In [ ]:
# Cell 4: Load Model (DeepSeek R1 7B) with Retry Logic
from unsloth import FastLanguageModel
import torch
import time

print(f"🧠 Loading: {MODEL_NAME}")
print("   (This may take 2-5 minutes...)")

# Retry logic for HuggingFace timeout
max_retries = 3
for attempt in range(max_retries):
    try:
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=MODEL_NAME,
            max_seq_length=MAX_SEQ_LENGTH,
            dtype=None,  # Auto-detect
            load_in_4bit=True,  # QLoRA quantization
        )
        break  # Success!
    except Exception as e:
        if "timeout" in str(e).lower() or "ReadTimeout" in str(type(e)):
            if attempt < max_retries - 1:
                wait_time = 30 * (attempt + 1)
                print(f"   ⚠️ Timeout (attempt {attempt+1}/{max_retries}), retrying in {wait_time}s...")
                time.sleep(wait_time)
            else:
                print(f"   ❌ Failed after {max_retries} attempts")
                raise
        else:
            raise

vram_used = torch.cuda.memory_allocated(0) / 1024**3
print(f"✅ Model loaded ({vram_used:.2f} GB VRAM)")
print(f"   DeepSeek R1 7B - optimized for reasoning with  tags")

In [ ]:
# Cell 5: Setup Chat Template (ChatML for DeepSeek R1)
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="chatml",
    mapping={"role": "role", "content": "content", "user": "user", "assistant": "assistant"},
    map_eos_token=True,
)

print("✅ ChatML template configured")
print("   Native format for DeepSeek R1 <think> reasoning")

In [ ]:
# Cell 6: Add LoRA Adapters (Recommended Settings)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention
        "gate_proj", "up_proj", "down_proj"       # MLP
    ],
    random_state=3407,
    use_rslora=False,
    use_gradient_checkpointing="unsloth",
)

print(f"✅ LoRA attached: rank={LORA_RANK}, alpha={LORA_ALPHA}")
print("   Targets: Full attention + MLP for reasoning capability")

In [ ]:
# Cell 7: Load Training Data
import json

print(f"📂 Loading data from {PROJECT_ROOT}")

if not TRAIN_FILE.exists():
    print(f"❌ Train file not found: {TRAIN_FILE}")
    print("   Make sure 'valora-training-data' dataset is attached!")
    raise FileNotFoundError(f"Missing: {TRAIN_FILE}")

with open(TRAIN_FILE, 'r', encoding='utf-8') as f:
    train_data = json.load(f)

with open(EVAL_FILE, 'r', encoding='utf-8') as f:
    eval_data = json.load(f)

print(f"✅ Loaded: {len(train_data):,} train + {len(eval_data):,} eval")

# Show sample structure
sample = train_data[0]
print(f"   Roles: {[m['role'] for m in sample['messages']]}")
print(f"   Has <think> tags: {'<think>' in str(sample)}")

In [ ]:
# Cell 8: Convert to Text Format
def convert_to_text_format(examples):
    """Convert messages format to conversation format"""
    formatted = []
    for ex in examples:
        messages = ex.get("messages", [])
        if not messages:
            continue
        
        conversation = []
        for msg in messages:
            role = msg.get("role", "")
            content = msg.get("content", "")
            
            # Handle different content formats
            if isinstance(content, str):
                text = content
            elif isinstance(content, list):
                text_parts = [item.get("text", "") for item in content 
                             if isinstance(item, dict) and item.get("type") == "text"]
                text = " ".join(text_parts)
            else:
                text = str(content)
            
            if text.strip():
                conversation.append({"role": role, "content": text})
        
        if conversation:
            formatted.append({"conversations": conversation})
    
    return formatted

train_formatted = convert_to_text_format(train_data)
eval_formatted = convert_to_text_format(eval_data)

print(f"✅ Converted: {len(train_formatted):,} train + {len(eval_formatted):,} eval")

In [ ]:
# Cell 9: Create HuggingFace Datasets
from datasets import Dataset

def create_dataset_with_text(formatted_data):
    """Create dataset with pre-tokenized text"""
    records = []
    for item in formatted_data:
        text = tokenizer.apply_chat_template(
            item["conversations"],
            tokenize=False,
            add_generation_prompt=False
        )
        records.append({"text": text})
    return Dataset.from_list(records)

train_dataset = create_dataset_with_text(train_formatted)
eval_dataset = create_dataset_with_text(eval_formatted)

print(f"✅ Datasets ready")
print(f"   Train: {len(train_dataset):,} | Eval: {len(eval_dataset):,}")
# Fix: Use select() to get a subset, then access the column
sample_texts = train_dataset.select(range(min(100, len(train_dataset))))['text']
avg_len = sum(len(t) for t in sample_texts) / len(sample_texts)
print(f"   Avg text length: {avg_len:.0f} chars")

In [ ]:
# Cell 10: Training Config (RECOMMENDED SETTINGS)
from trl import SFTConfig
from unsloth import is_bfloat16_supported

# Calculate total steps for progress tracking
total_steps = (len(train_dataset) // GRAD_ACCUM) * NUM_EPOCHS

training_config = SFTConfig(
    output_dir=str(CHECKPOINT_DIR),
    
    # Batch settings
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    
    # Training schedule
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    
    # Precision
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    
    # Memory optimization
    gradient_checkpointing=True,
    optim="adamw_8bit",
    max_grad_norm=MAX_GRAD_NORM,
    
    # Logging
    logging_steps=10,
    logging_first_step=True,
    
    # CHECKPOINT SETTINGS (CRITICAL FOR KAGGLE)
    save_strategy="steps",
    save_steps=100,  # Save every 100 steps
    save_total_limit=5,  # Keep last 5 checkpoints
    
    # Evaluation
    eval_strategy="steps",
    eval_steps=200,
    
    # Resume capability
    resume_from_checkpoint=True,
    load_best_model_at_end=False,  # We'll manually select
    
    # Other
    seed=3407,
    report_to="none",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
)

print(f"✅ Training config ready")
print(f"   Epochs: {NUM_EPOCHS} | Steps: ~{total_steps}")
print(f"   LR: {LEARNING_RATE} | Batch: {GRAD_ACCUM} (effective)")
print(f"   Checkpoints: Every 100 steps → {CHECKPOINT_DIR}")

In [ ]:
# Cell 11: Initialize Trainer with Checkpoint Detection
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=training_config,
    dataset_text_field="text",
)

# Detect existing checkpoints
checkpoints = sorted(CHECKPOINT_DIR.glob("checkpoint-*"), 
                    key=lambda x: int(x.name.split("-")[1]) if x.name.split("-")[1].isdigit() else 0)

if checkpoints:
    latest = checkpoints[-1]
    step_num = latest.name.split("-")[1]
    print(f"🔄 Found {len(checkpoints)} checkpoint(s)")
    print(f"   Latest: {latest.name} (step {step_num})")
    print(f"   Training will RESUME from this checkpoint")
else:
    print("🆕 No checkpoints found - starting fresh")

print(f"\n📊 Dataset: {len(train_dataset):,} train + {len(eval_dataset):,} eval")

In [ ]:
# Cell 12: TRAIN WITH ROBUST CHECKPOINT HANDLING
import gc
import torch
from pathlib import Path

print("=" * 60)
print("🚀 STARTING FINE-TUNING")
print("=" * 60)

# Detect latest checkpoint for resume
checkpoints = sorted(CHECKPOINT_DIR.glob("checkpoint-*"), 
                    key=lambda x: int(x.name.split("-")[1]) if x.name.split("-")[1].isdigit() else 0)
resume_from = str(checkpoints[-1]) if checkpoints else None

if resume_from:
    print(f"🔄 RESUMING FROM: {Path(resume_from).name}")
else:
    print("🆕 STARTING FRESH TRAINING")

print("=" * 60)
print("💡 KAGGLE PERSISTENCE ENABLED:")
print("   - Checkpoints saved to /kaggle/working/checkpoints/")
print("   - Auto-resume on kernel restart")
print("   - Re-run this cell to continue training")
print("=" * 60)
print()

# Clean memory before training
gc.collect()
torch.cuda.empty_cache()

# Start progress tracker
progress_tracker.start(total_steps=total_steps)

try:
    # Main training loop with auto-resume
    result = trainer.train(resume_from_checkpoint=resume_from)
    
    print("\n" + "=" * 60)
    print("✅ TRAINING COMPLETED SUCCESSFULLY!")
    print("=" * 60)
    print(f"   Final loss: {result.training_loss:.4f}")
    print(f"   Total steps: {result.global_step}")
    print(f"   Checkpoints: {CHECKPOINT_DIR}")
    
    # Save final model
    print("\n💾 Saving final model...")
    trainer.save_model(str(OUTPUT_DIR))
    tokenizer.save_pretrained(str(OUTPUT_DIR))
    print(f"   Saved to: {OUTPUT_DIR}")
    
    progress_tracker.stop()
    
except KeyboardInterrupt:
    print("\n" + "=" * 60)
    print("⚠️ TRAINING INTERRUPTED")
    print("=" * 60)
    # Find latest checkpoint after interrupt
    checkpoints = sorted(CHECKPOINT_DIR.glob("checkpoint-*"))
    if checkpoints:
        print(f"   Last checkpoint: {checkpoints[-1].name}")
        print("   Re-run this cell to RESUME training")
    progress_tracker.stop()

except Exception as e:
    print("\n" + "=" * 60)
    print(f"❌ ERROR: {type(e).__name__}")
    print("=" * 60)
    print(f"   {str(e)[:200]}")
    checkpoints = sorted(CHECKPOINT_DIR.glob("checkpoint-*"))
    if checkpoints:
        print(f"   Last checkpoint: {checkpoints[-1].name}")
        print("   Re-run this cell to RESUME from checkpoint")
    progress_tracker.stop()
    raise

In [ ]:
# Cell 13: Export to GGUF for Ollama
print("📦 EXPORTING MODEL TO GGUF")
print("=" * 60)

# Check if training completed
if not (OUTPUT_DIR / "adapter_config.json").exists():
    print("⚠️ No final model found. Run training cell first.")
    print("   Or export from latest checkpoint:")
    checkpoints = sorted(CHECKPOINT_DIR.glob("checkpoint-*"))
    if checkpoints:
        print(f"   Latest: {checkpoints[-1]}")
else:
    print("✅ Final model found, proceeding with export...")
    
    # Export to GGUF (Q4_K_M quantization)
    try:
        model.save_pretrained_gguf(
            str(OUTPUT_DIR / "gguf"),
            tokenizer,
            quantization_method="q4_k_m"
        )
        print("✅ GGUF export complete!")
        print(f"   Location: {OUTPUT_DIR / 'gguf'}")
    except Exception as e:
        print(f"⚠️ GGUF export failed: {e}")
        print("   You can export manually after downloading the LoRA adapters")

In [ ]:
# Cell 14: Create Modelfile for Ollama
modelfile_content = '''FROM valora-deepseek-r1.gguf

PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER top_k 40
PARAMETER repeat_penalty 1.1
PARAMETER num_ctx 3072

SYSTEM """You are Valora AI, an enterprise real estate intelligence assistant for Bangalore.

## Core Capabilities
- Deep reasoning with <think> tags for complex analysis
- Financial modeling: ROI, IRR, rental yield, cap rate
- Risk assessment and portfolio optimization
- Market intelligence and trend analysis

## Guidelines
- Use Indian formats: ₹ Lakhs/Crores, sqft, BHK
- Show reasoning process with <think>...</think> tags
- Never invent data - only use provided facts
- Bangalore/Bengaluru focus exclusively
- Professional terminology: NOI, GRM, Cap Rate, LTV"""
'''

with open(OUTPUT_DIR / "Modelfile.valora", 'w') as f:
    f.write(modelfile_content)

print("✅ Modelfile created")
print(f"   Location: {OUTPUT_DIR / 'Modelfile.valora'}")

In [ ]:
# Cell 15: Final Summary & Download Instructions
print("=" * 60)
print("🎉 VALORA AI FINE-TUNING COMPLETE")
print("=" * 60)

print(f"\n📊 TRAINING DETAILS:")
print(f"   Model: DeepSeek R1 7B (Qwen-based)")
print(f"   Dataset: 8,061 examples (v3.3)")
print(f"   Categories: 37 intent types")
print(f"   LoRA: rank={LORA_RANK}, alpha={LORA_ALPHA}")

print(f"\n🎯 CAPABILITIES TRAINED:")
print("   ✅ Financial: ROI, IRR, rental yield, cap rate")
print("   ✅ Portfolio: Multi-property optimization")
print("   ✅ Risk: SWOT, terrain, compliance")
print("   ✅ Reasoning: <think> tags for step-by-step analysis")
print("   ✅ Refusal: 13+ out-of-scope scenarios")
print("   ✅ Simulation: What-if scenarios with deltas")
print("   ✅ Storyboard: 3D camera sequences & narration")
print("   ✅ UI Control: Panel/tab/mode actions")

print(f"\n📁 OUTPUT FILES:")
print(f"   Checkpoints: {CHECKPOINT_DIR}")
print(f"   Final Model: {OUTPUT_DIR}")

# List files to download
print(f"\n📥 DOWNLOAD THESE FILES:")
for f in OUTPUT_DIR.glob("*"):
    size = f.stat().st_size / 1024**2 if f.is_file() else 0
    print(f"   - {f.name}" + (f" ({size:.1f} MB)" if size > 0 else ""))

print("=" * 60)

In [ ]:
# Cell 16: Deployment Instructions
print("=" * 60)
print("📦 DEPLOYMENT GUIDE")
print("=" * 60)

print("\n1️⃣ DOWNLOAD FROM KAGGLE OUTPUT:")
print("   After training completes, download from /kaggle/working/:")
print("   - final_model/ (LoRA adapters)")
print("   - gguf/valora-*.gguf (if exported)")
print("   - Modelfile.valora")

print("\n2️⃣ LOCAL OLLAMA SETUP:")
print("   # Option A: Use GGUF directly")
print("   ollama create valora -f Modelfile.valora")
print("   ollama run valora")

print("\n3️⃣ INTEGRATE WITH VALORA BACKEND:")
print("   Update backend/llm_config.json:")
print("   {")
print("     \"provider\": \"ollama\",")
print("     \"model\": \"valora\",")
print("     \"base_url\": \"http://localhost:11434\"")
print("   }")

print("\n4️⃣ TEST THE MODEL:")
print("   curl http://localhost:11434/api/generate -d")
print("   '{\"model\":\"valora\",\"prompt\":\"Analyze Koramangala\"}'")

print("\n🔄 TO RESUME INTERRUPTED TRAINING:")
print("   1. Re-attach the same Kaggle dataset")
print("   2. Run cells 1-11 (setup)")
print("   3. Run cell 12 (training) - it auto-resumes!")

print("\n✅ Your Valora AI with DeepSeek R1 reasoning is ready!")
print("=" * 60)